# 04 — Arm 3: Frozen retrieval shortlist + LLM selection (Colab)

Colab variant of the Arm 3 notebook. Set `QUICK_SMOKE_TEST = True` for a fast CPU-only correctness run that exercises the full prompt-building and parsing pipeline without downloading the seq2seq model. Set it to `False` on a GPU runtime for the full frozen run. All substantive logic lives in `src/amlh/arm3_llm.py`.

In [ ]:
QUICK_SMOKE_TEST = True

In [ ]:
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer

from amlh import arm3_llm as a3
from amlh.config import ARTEFACTS_DIR, FIGURES_DIR, HYPERPARAMETERS, SEED, set_seed

## 1. Load splits from `artefacts/`

In [ ]:
set_seed()

fit = pd.read_csv(ARTEFACTS_DIR / "split_fit.csv")
val = pd.read_csv(ARTEFACTS_DIR / "split_val.csv")

if QUICK_SMOKE_TEST:
    fit = fit.groupby("disease", group_keys=False)[fit.columns].apply(lambda g: g.head(1)).sample(
        n=min(40, len(fit)), random_state=44
    ).reset_index(drop=True)
    val = val.sample(n=min(20, len(val)), random_state=44).reset_index(drop=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"fit={len(fit)} ({fit.disease.nunique()} classes) | val={len(val)} ({val.disease.nunique()} classes)")
print(f"device={device} | gpu={gpu_name}")

shortlist_k = HYPERPARAMETERS.shortlist_k
llm_temperature = HYPERPARAMETERS.llm_temperature
arm3_model_name = a3.selected_model_name(HYPERPARAMETERS)
n_shots = 2
prompt_modes = ["zero_shot", "few_shot", "cot"]
print({"shortlist_k": shortlist_k, "llm_temperature": llm_temperature, "prompt_mode": None, "n_shots": n_shots, "arm3_model_name": arm3_model_name})

## 2. Prompt budget check

Prompt lengths are measured before any condition is run, using the model tokenizer, and the truncation rate at 512 is printed explicitly.

In [ ]:
shortlist_rankings, _ = a3.build_shortlist_ranking(fit, val, HYPERPARAMETERS, depth=shortlist_k)
examples = a3.build_examples(fit, n=n_shots, seed=SEED)
tokeniser = AutoTokenizer.from_pretrained(arm3_model_name)

prompts_by_mode = {}
budget_rows = []
for mode in prompt_modes:
    prompts = a3.build_prompts_for_condition(
        val, shortlist_rankings, mode, examples=examples if mode == "few_shot" else None
    )
    prompts_by_mode[mode] = prompts
    summary = a3.prompt_token_lengths(prompts, tokeniser, max_length=512)
    budget_rows.append({
        "condition": mode,
        "min_tokens": summary["min"],
        "median_tokens": summary["median"],
        "mean_tokens": summary["mean"],
        "p95_tokens": summary["p95"],
        "max_tokens": summary["max"],
        "truncation_rate_512": summary["truncation_rate"],
    })

budget_df = pd.DataFrame(budget_rows)
print(budget_df.to_string(index=False))

## 3. Run all prompt conditions

The smoke-test path uses a deterministic fake generator so the notebook can validate the full orchestration without downloading Flan-T5-large. The full run loads the model and uses greedy decoding.

In [ ]:
if QUICK_SMOKE_TEST:
    generator = lambda prompt: "1"
    model = None
    model_tokeniser = tokeniser
else:
    model_tokeniser, model = a3.load_generator(arm3_model_name, device=device)

condition_frames = {}
condition_rows = []
condition_prompts = {}

for mode in prompt_modes:
    start = time.perf_counter()
    pred_df, metrics, prompts = a3.run_condition(
        fit,
        val,
        HYPERPARAMETERS,
        mode=mode,
        tokeniser=None if QUICK_SMOKE_TEST else model_tokeniser,
        model=None if QUICK_SMOKE_TEST else model,
        device=None if QUICK_SMOKE_TEST else device,
        generator=generator if QUICK_SMOKE_TEST else None,
        examples=examples if mode == "few_shot" else None,
        shortlist_depth=shortlist_k,
    )
    metrics["wall_clock_sec"] = time.perf_counter() - start
    condition_frames[mode] = pred_df
    condition_rows.append(metrics)
    condition_prompts[mode] = prompts

condition_df = pd.DataFrame(condition_rows)
print(condition_df.to_string(index=False))

## 4. Pairwise McNemar and selection

In [ ]:
mcnemar_df = a3.pairwise_condition_mcnemar(condition_frames)
selected_mode, tie_break_fired = a3.select_prompt_mode(condition_df, mcnemar_df)
print(mcnemar_df.to_string(index=False))
print({"selected_mode": selected_mode, "tie_break_fired": tie_break_fired})

predictions_df = pd.concat(condition_frames.values(), ignore_index=True)
predictions_df.to_csv(ARTEFACTS_DIR / "arm3_val_predictions.csv", index=False)
condition_df.to_csv(ARTEFACTS_DIR / "arm3_prompt_conditions.csv", index=False)
mcnemar_df.to_csv(ARTEFACTS_DIR / "arm3_condition_mcnemar.csv", index=False)
with open(ARTEFACTS_DIR / "arm3_prompts.txt", "w", encoding="utf-8") as f:
    f.write(a3.build_prompt_table(condition_prompts))

print({"shortlist_k": shortlist_k, "llm_temperature": llm_temperature, "prompt_mode": selected_mode, "n_shots": n_shots, "arm3_model_name": arm3_model_name})

## 5. Zip artefacts for download

In [ ]:
import zipfile

zip_path = Path("arm3_colab_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in (ARTEFACTS_DIR, FIGURES_DIR):
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=path.relative_to(ARTEFACTS_DIR.parent))
print(f"wrote {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print(f"download unavailable: {exc}")